<a href="https://colab.research.google.com/github/shims79757-lang/Elevance-Skills-Projects/blob/main/Google_Play_Store_App_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from datetime import datetime
from zoneinfo import ZoneInfo
from matplotlib.path import Path

In [7]:
apps = pd.read_csv("/content/googleplaystore.csv")
reviews = pd.read_csv("/content/googleplaystore_user_reviews.csv")

print("Apps dataset:", apps.shape)
print("Reviews dataset:", reviews.shape)

apps.head()

Apps dataset: (10841, 13)
Reviews dataset: (64295, 5)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [8]:
# Work on a copy of the original dataset
data = apps.copy()

# Convert Rating to numbers
data["Rating"] = pd.to_numeric(
    data["Rating"],
    errors="coerce"
)

# Convert Reviews to numbers
data["Reviews"] = pd.to_numeric(
    data["Reviews"],
    errors="coerce"
)

# Remove commas and + signs from Installs
data["Installs"] = (
    data["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

# Convert Installs to numbers
data["Installs"] = pd.to_numeric(
    data["Installs"],
    errors="coerce"
)

data[["App", "Rating", "Reviews", "Installs"]].head()

,App,Rating,Reviews,Installs
0,Photo Editor & Candy Camera & Grid & ScrapBook,4.1,159.0,10000.0
1,Coloring book moana,3.9,967.0,500000.0
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",4.7,87510.0,5000000.0
3,Sketch - Draw & Paint,4.5,215644.0,50000000.0
4,Pixel Draw - Number Art Coloring Book,4.3,967.0,100000.0


In [9]:
def convert_size(size):

    size = str(size).strip()

    if size.endswith("M"):
        return float(size[:-1])

    elif size.endswith("k"):
        return float(size[:-1]) / 1024

    else:
        return np.nan


data["Size_MB"] = data["Size"].apply(convert_size)

data[["App", "Size", "Size_MB"]].head()

,App,Size,Size_MB
0,Photo Editor & Candy Camera & Grid & ScrapBook,19M,19.0
1,Coloring book moana,14M,14.0
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",8.7M,8.7
3,Sketch - Draw & Paint,25M,25.0
4,Pixel Draw - Number Art Coloring Book,2.8M,2.8


In [10]:
app_subjectivity = (
    reviews
    .groupby("App")["Sentiment_Subjectivity"]
    .mean()
    .reset_index()
)

app_subjectivity.head()

,App,Sentiment_Subjectivity
0,10 Best Foods for You,0.495455
1,104 找工作 - 找工作 找打工 找兼職 履歷健檢 履歷診療室,0.545516
2,11st,0.443957
3,1800 Contacts - Lens Store,0.591098
4,1LINE – One Line with One Touch,0.557315


In [11]:
data = data.merge(
    app_subjectivity,
    on="App",
    how="left"
)

data[
    [
        "App",
        "Category",
        "Rating",
        "Reviews",
        "Installs",
        "Size_MB",
        "Sentiment_Subjectivity"
    ]
].head()

,App,Category,Rating,Reviews,Installs,Size_MB,Sentiment_Subjectivity
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159.0,10000.0,19.0,NaN
1,Coloring book moana,ART_AND_DESIGN,3.9,967.0,500000.0,14.0,0.64154
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510.0,5000000.0,8.7,NaN
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644.0,50000000.0,25.0,NaN
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967.0,100000.0,2.8,NaN


In [12]:
categories = [
    "GAME",
    "BEAUTY",
    "BUSINESS",
    "COMICS",
    "COMMUNICATION",
    "DATING",
    "ENTERTAINMENT",
    "SOCIAL",
    "EVENTS"
]

data = data[
    data["Category"].isin(categories)
].copy()

print(data["Category"].value_counts())

Category
GAME             1144
BUSINESS          460
COMMUNICATION     387
SOCIAL            295
DATING            234
ENTERTAINMENT     149
EVENTS             64
COMICS             60
BEAUTY             53
Name: count, dtype: int64


In [13]:
data = data[
    (data["Rating"] > 3.5) &
    (data["Installs"] > 50000) &
    (data["Reviews"] > 500) &
    (data["Size_MB"] >= 10) &
    (data["Size_MB"] <= 100) &
    (data["Sentiment_Subjectivity"] > 0.5)
].copy()

print("Apps remaining:", len(data))

data.head()

Apps remaining: 106


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Size_MB,Sentiment_Subjectivity
104,Hairstyles step by step,BEAUTY,4.6,4369.0,14M,100000.0,Free,0,Everyone,Beauty,"July 25, 2018",1.9,4.0.3 and up,14.0,0.591160
105,Filters for Selfie,BEAUTY,4.3,8572.0,25M,1000000.0,Free,0,Everyone,Beauty,"May 10, 2018",1.1.0,4.0 and up,25.0,0.527895
198,Google Primer,BUSINESS,4.4,62272.0,18M,10000000.0,Free,0,Everyone,Business,"June 26, 2018",3.550.2,4.1 and up,18.0,0.675000
247,Crew - Free Messaging and Scheduling,BUSINESS,4.6,4159.0,48M,500000.0,Free,0,Everyone,Business,"July 20, 2018",6.1.2,4.0.3 and up,48.0,0.599838
248,Asana: organize team projects,BUSINESS,4.3,20815.0,10M,1000000.0,Free,0,Everyone,Business,"July 26, 2018",6.4.4,5.0 and up,10.0,0.522767


In [14]:
data = data[
    ~data["App"].str.contains(
        "s",
        case=False,
        na=False
    )
].copy()

print("Apps remaining after removing S/s:", len(data))

Apps remaining after removing S/s: 34


In [15]:
data = data.drop_duplicates(
    subset=["App", "Category"]
)

data = data.reset_index(drop=True)

In [16]:
translations = {
    "BEAUTY": "सुंदरता",
    "BUSINESS": "வணிகம்",
    "DATING": "Partnersuche"
}

data["Display_Category"] = data["Category"].replace(
    translations
)

data[["Category", "Display_Category"]].drop_duplicates()

,Category,Display_Category
0,BUSINESS,வணிகம்
1,COMMUNICATION,COMMUNICATION
3,DATING,Partnersuche
7,ENTERTAINMENT,ENTERTAINMENT
9,GAME,GAME


In [17]:
def get_outliers(group):

    # SIZE IQR

    size_q1 = group["Size_MB"].quantile(0.25)
    size_q3 = group["Size_MB"].quantile(0.75)

    size_iqr = size_q3 - size_q1

    size_lower = size_q1 - (1.5 * size_iqr)
    size_upper = size_q3 + (1.5 * size_iqr)


    # RATING IQR

    rating_q1 = group["Rating"].quantile(0.25)
    rating_q3 = group["Rating"].quantile(0.75)

    rating_iqr = rating_q3 - rating_q1

    rating_lower = rating_q1 - (1.5 * rating_iqr)
    rating_upper = rating_q3 + (1.5 * rating_iqr)


    # Find apps outside the limits

    outliers = group[
        (group["Size_MB"] < size_lower) |
        (group["Size_MB"] > size_upper) |
        (group["Rating"] < rating_lower) |
        (group["Rating"] > rating_upper)
    ]

    return outliers

In [18]:
outliers = (
    data
    .groupby("Category", group_keys=False)
    .apply(get_outliers)
    .reset_index(drop=True)
)

print("Number of outliers:", len(outliers))

outliers[
    [
        "App",
        "Display_Category",
        "Size_MB",
        "Rating"
    ]
]

Number of outliers: 0


/tmp/ipykernel_1248/1633783915.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(get_outliers)


,App,Display_Category,Size_MB,Rating


In [19]:
def make_hexagons(data, x_bins=9, y_bins=7):

    hexagons = []

    if data.empty:
        return hexagons


    # Chart limits

    min_size = data["Size_MB"].min()
    max_size = data["Size_MB"].max()

    min_rating = data["Rating"].min()
    max_rating = data["Rating"].max()


    # Prevent problems if values are identical

    if min_size == max_size:
        min_size -= 1
        max_size += 1

    if min_rating == max_rating:
        min_rating -= 0.1
        max_rating += 0.1


    # Width and height of each region

    width = (max_size - min_size) / x_bins
    height = (max_rating - min_rating) / y_bins


    # Create rows

    for row in range(y_bins):

        center_y = min_rating + (row + 0.5) * height

        # Shift alternate rows
        if row % 2 == 0:
            offset = 0
        else:
            offset = width / 2


        for column in range(x_bins):

            center_x = (
                min_size +
                (column + 0.5) * width +
                offset
            )


            # Coordinates of hexagon

            angles = np.linspace(
                0,
                2 * np.pi,
                7
            )

            hex_x = (
                center_x +
                width * 0.58 * np.cos(angles)
            )

            hex_y = (
                center_y +
                height * 0.58 * np.sin(angles)
            )


            # Check which apps are inside

            polygon = Path(
                np.column_stack(
                    (hex_x, hex_y)
                )
            )

            points = data[
                ["Size_MB", "Rating"]
            ].values

            inside = polygon.contains_points(points)

            apps_inside = data[inside]


            # Only save non-empty regions

            if len(apps_inside) > 0:

                hexagons.append({

                    "x": hex_x,

                    "y": hex_y,

                    "count": len(apps_inside),

                    "average_installs":
                        apps_inside["Installs"].mean(),

                    "apps":
                        ", ".join(
                            apps_inside["App"].head(5)
                        )
                })


    return hexagons

In [20]:
hexagons = make_hexagons(data)

print(
    "Number of hexagonal regions containing apps:",
    len(hexagons)
)

Number of hexagonal regions containing apps: 15


In [21]:
fig = make_subplots(

    rows=2,
    cols=2,

    column_widths=[0.82, 0.18],

    row_heights=[0.18, 0.82],

    horizontal_spacing=0.03,

    vertical_spacing=0.03,

    specs=[
        [{"type": "xy"}, None],
        [{"type": "xy"}, {"type": "xy"}]
    ]
)

In [22]:
if len(hexagons) > 0:

    install_values = [
        h["average_installs"]
        for h in hexagons
    ]

    minimum = min(install_values)
    maximum = max(install_values)


    for h in hexagons:

        if maximum != minimum:

            intensity = (
                h["average_installs"] - minimum
            ) / (
                maximum - minimum
            )

        else:
            intensity = 0.5


        # Create increasing colour intensity

        red = int(235 - intensity * 90)
        green = int(210 - intensity * 130)
        blue = int(235 - intensity * 40)

        colour = (
            f"rgba({red},"
            f"{green},"
            f"{blue},0.85)"
        )


        fig.add_trace(

            go.Scatter(

                x=h["x"],
                y=h["y"],

                mode="lines",

                fill="toself",

                fillcolor=colour,

                line=dict(
                    color="white",
                    width=1
                ),

                hovertemplate=(
                    "<b>Hexagonal Region</b><br>"
                    f"Number of Apps: {h['count']}<br>"
                    f"Average Installs: "
                    f"{h['average_installs']:,.0f}<br>"
                    f"Apps: {h['apps']}"
                    "<extra></extra>"
                ),

                showlegend=False
            ),

            row=2,
            col=1
        )

In [23]:
game_apps = data[
    data["Category"] == "GAME"
]

fig.add_trace(

    go.Scatter(

        x=game_apps["Size_MB"],

        y=game_apps["Rating"],

        mode="markers",

        name="Game Apps",

        marker=dict(
            color="deeppink",
            size=10,
            line=dict(
                color="white",
                width=1
            )
        ),

        text=game_apps["App"],

        customdata=np.column_stack(
            (
                game_apps["Installs"],
                game_apps["Reviews"]
            )
        ),

        hovertemplate=(
            "<b>%{text}</b><br>"
            "Size: %{x:.1f} MB<br>"
            "Rating: %{y:.2f}<br>"
            "Installs: %{customdata[0]:,.0f}<br>"
            "Reviews: %{customdata[1]:,.0f}"
            "<extra></extra>"
        )
    ),

    row=2,
    col=1
)

In [24]:
if len(outliers) > 0:

    fig.add_trace(

        go.Scatter(

            x=outliers["Size_MB"],

            y=outliers["Rating"],

            mode="markers+text",

            name="IQR Outliers",

            text=outliers["App"],

            textposition="top center",

            marker=dict(
                size=12,
                color="black",
                symbol="diamond"
            ),

            hovertemplate=(
                "<b>%{text}</b><br>"
                "Size: %{x:.1f} MB<br>"
                "Rating: %{y:.2f}"
                "<extra></extra>"
            )
        ),

        row=2,
        col=1
    )

In [25]:
fig.add_trace(

    go.Histogram(

        x=data["Size_MB"],

        nbinsx=15,

        name="Size Distribution",

        marker=dict(
            color="lightpink"
        ),

        showlegend=False
    ),

    row=1,
    col=1
)

In [26]:
fig.add_trace(

    go.Histogram(

        y=data["Rating"],

        nbinsy=12,

        name="Rating Distribution",

        marker=dict(
            color="lightpink"
        ),

        showlegend=False
    ),

    row=2,
    col=2
)

In [27]:
fig.update_xaxes(
    title_text="App Size (MB)",
    row=2,
    col=1
)

fig.update_yaxes(
    title_text="Rating",
    row=2,
    col=1
)


fig.update_layout(

    title=dict(

        text=(
            "<b>Google Play Store App Analysis</b>"
            "<br>"
            "<sup>"
            "Relationship Between App Size and Rating "
            "| Colour Intensity = Average Installs"
            "</sup>"
        ),

        x=0.5
    ),

    template="plotly_white",

    height=750,

    hovermode="closest",

    margin=dict(
        l=70,
        r=50,
        t=110,
        b=70
    ),

    legend=dict(
        orientation="h",
        y=1.02,
        x=1,
        xanchor="right"
    )
)

In [28]:
current_time = datetime.now(
    ZoneInfo("Asia/Kolkata")
)

print(
    "Current IST:",
    current_time.strftime(
        "%d %B %Y, %I:%M %p"
    )
)


if 17 <= current_time.hour < 19:

    fig.show()

else:

    print(
        "Visualization unavailable.\n"
        "This graph can only be viewed "
        "between 5:00 PM and 7:00 PM IST."
    )

Current IST: 24 September 2026, 06:24 PM
